# Predicting Harvest and Post-Harvest Crop Loss in Nigeria

## 1. Project goal

Harvest and post-harvest crop losses can reduce food availability and farmers' income. This project investigates whether characteristics recorded in the Nigeria National Agricultural Sample Survey 2023 can help identify crop records where losses were reported.

### Research question

How well can crop, household, and farm characteristics predict whether a household reports harvest or post-harvest loss for a crop during the main agricultural season?

### Objective

Build and evaluate binary classification models that distinguish between reported loss and no reported loss.

### Scope

The initial investigation will use main-season data and focus on whether loss occurred. Predicting the amount of loss is a possible extension.

The intended prediction point is before harvest. Candidate predictors must therefore be assessed for whether their information would be available at that point.

### Planned comparison

Does adding household and farm characteristics improve predictions beyond using crop type and location alone?

### Evaluation

Models will be compared using precision, recall, and F1-score for the loss class, supported by a confusion matrix. A dummy classifier will provide a minimum benchmark, followed by logistic regression and tree-based models.

The investigation concerns predictive relationships, not proof that a particular characteristic causes crop loss.

## 2. Initial data inspection

### 2.1 Check the working directory and source files

Check that the active Python environment can access the project data. All paths are relative to the project root. The original survey files will remain unchanged.

In [15]:
from pathlib import Path

import pandas as pd
from IPython.display import display

DATA_DIR = Path("data/raw")
HARVEST_FILE = DATA_DIR / "5_Harvest_mainSeason.dta"

print("Working directory:", Path.cwd())

if not HARVEST_FILE.is_file():
  raise FileNotFoundError(
    f"Cannot find {HARVEST_FILE.resolve()}. "
    "Check the working directory and selected kernel"
  )

print("Raw data files:", len(list(DATA_DIR.glob("*.dta"))))

Working directory: /Users/mac/Documents/HIS/Courses/Introduction to Data Science A1N/Assignments/nass-postharvest-project
Raw data files: 21


The project data directory is accessible from the active environment and contains 21 Stata files. I will begin with the main-season harvest file because it contains the reported-loss outcome.

### 2.2 Load the main-season harvest data

I load the harvest file and inspect its dimensions and first few rows. This establishes the size and basic structure of the starting table.

In [18]:
harvest = pd.read_stata(HARVEST_FILE)

print("Rows:", f"{harvest.shape[0]:,}")
print("Colowns:", harvest.shape[1])

display(harvest.head())

Rows: 334,133
Colowns: 68


,hhid,zone_id,state_id,lga,sector,final_weight,s01b_harvestedCrops__id,s01b_q02,s01b_q03a,s01b_q03c,...,s01b_q19c,s01b_q20,s01b_q21a,s01b_q21c,s01b_q22,s01b_q23,s01b_q24,s01b_q25,s01b_q26,s01b_q27
0,326074.0,1.0,NASARAWA,TOTO,URBAN,43.316891,RICE,No,0.000000,NaN,...,NaN,Yes,158.000,Unshelled (dry),100.0,None (0%),None (0%),None (0%),None (0%),None (0%)
1,342175.0,1.0,PLATEAU,SHENDAM,RURAL,185.124704,RICE,Yes,1896.000000,Unshelled (dry),...,NaN,No,0.000,NaN,None (0%),None (0%),None (0%),None (0%),None (0%),None (0%)
2,274109.0,3.0,KEBBI,BAGUDO,URBAN,153.188059,GUINEA CORN/SORGHUM,Yes,1500.000000,NaN,...,NaN,Yes,510.000,NaN,10.0,70.0,5.0,None (0%),None (0%),15.0
3,347050.0,2.0,TARABA,JALINGO,RURAL,89.243114,BEANS/COWPEA,Yes,795.555556,Unshelled (dry),...,NaN,No,0.000,NaN,None (0%),None (0%),None (0%),None (0%),None (0%),None (0%)
4,545170.0,4.0,ANAMBRA,IDEMILI NORTH,URBAN,390.568099,MELON/EGUSI,Yes,2.110000,Unshelled (fresh),...,NaN,Yes,2.462,Shelled (dry),50.0,None (0%),50.0,None (0%),None (0%),None (0%)


The main-season harvest table contains 334,133 records and 68 variables. The preview shows household identifiers, geographical information, crop
identifiers, and questions about harvest use and losses.

### 2.3 Inspect the prediction target

The proposed target is `s01b_q11`, which records whether losses occurred during/after harvest, post-harvest activities, transport, or storage.

I check its response categories, missing values, and class distribution to assess whether it is suitable for binary classification.

In [20]:
TARGET = "s01b_q11"

print("Missing target values:", harvest[TARGET].isna().sum())

target_summary = harvest[TARGET].value_counts(dropna=False).to_frame("records")

target_summary["percent"] = (
  target_summary["records"] / len(harvest) * 100
).round(2)

display(target_summary)

Missing target values: 0


,records,percent
s01b_q11,,
No,295686,88.49
Yes,38447,11.51


- The target contains two response categories with no missing values. Loss was reported for 38,447 records (11.51%), while 295,686 records (88.49%) reported no loss.

- The classes are imbalanced. Predicting no loss for every record would achieve 88.49% accuracy while identifying none of the reported losses. Accuracy alone will therefore be insufficient for evaluating the models.

- These percentages describe unweighted crop records in this table; they are not national estimates of the percentage of households experiencing loss.

### 2.4 Check the unit of observation

A household may report several crops. I check whether the combination of household ID and harvested-crop ID uniquely identifies each record.

This determines if there are other tables that must be joined and helps establish an appropriate validation strategy.

In [26]:
ROW_KEY = ["hhid", "s01b_harvestedCrops__id"]

print("Distict households:", f"{harvest['hhid'].nunique():,}")

print("\nMissing values in the key:")
display(harvest[ROW_KEY].isna().sum())

print(
  "Duplicate household-crop keys:",
  harvest.duplicated(ROW_KEY).sum()
)

Distict households: 116,406

Missing values in the key:


hhid                       0
s01b_harvestedCrops__id    0
dtype: int64

Duplicate household-crop keys: 0


- The table contains 116,406 distinct households. None of the key columns has missing values, and no household-crop key combinations are duplicated.

- A row can therefore be treated as one household-crop record within the main-season harvest table. Multiple records can belong to the same household.

- Other data must be made compatible before joining. During model validation, all records from a household will be kept in the same partition to prevent household overlap between training and testing.

## 3. Initial findings and next decisions

The research question concerns predicting reported loss for a household–crop record during the main agricultural season.

The harvest table provides a complete binary target, but the classes are imbalanced. Household–crop identifiers are complete and unique.

Data management is not yet complete. The next tasks are to:

- Inspect the meanings and availability of candidate predictors.
- Investigate missing values and special numeric codes.
- Select relevant household, crop, and plot information.
- Decide how to aggregate and join these sources without duplicating records.
- Prepare and save a documented modelling table.

Loss amounts, loss percentages, and reasons for loss will not be used to predict loss occurrence because they reveal information about the outcome.